# 03 – Feature Engineering

**Projekt:** WealthScope AI  
**Kontext:** QUA3CK / Big-Data / Machine-Learning / Streamlit-App  
**Datenbasis:** Kaggle Stock/ETF Dataset, lokal verarbeitet  
**Hinweis:** Diese Notebooks dienen der wissenschaftlichen Nachvollziehbarkeit. Sie ersetzen keine Finanzberatung.

## Ziel

Dieses Notebook erklärt und prüft die verwendeten Features:

- `daily_return`
- `return_5d`
- `return_20d`
- `ma_20`, `ma_50`, `ma_200`
- `ma_*_distance`
- `volatility_20d`
- `drawdown`
- `future_return_20d`
- `target_20d`

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

PARQUET_PATH = DATA_DIR / "wealthscope_features.parquet"
CSV_PATH = DATA_DIR / "wealthscope_features.csv"

def load_features():
    if PARQUET_PATH.exists():
        df = pd.read_parquet(PARQUET_PATH)
        source = "REAL_PARQUET"
    elif CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH)
        source = "REAL_CSV"
    else:
        raise FileNotFoundError("Kein Feature-Datensatz gefunden. Erwartet wealthscope_features.parquet oder .csv")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return df, source

df, source = load_features()

print("Datenquelle:", source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
feature_groups = {
    "Preis": ["open", "high", "low", "close", "volume"],
    "Rendite": ["daily_return", "return_5d", "return_20d"],
    "Trend": ["ma_20", "ma_50", "ma_200", "ma_20_distance", "ma_50_distance", "ma_200_distance"],
    "Risiko": ["volatility_20d", "rolling_high", "drawdown"],
    "Zielvariable": ["future_return_20d", "target_20d"],
}

rows = []
for group, cols in feature_groups.items():
    for col in cols:
        rows.append({
            "Gruppe": group,
            "Feature": col,
            "Vorhanden": col in df.columns,
            "Fehlende Werte %": round(df[col].isna().mean() * 100, 2) if col in df.columns else None,
        })

pd.DataFrame(rows)

In [ ]:
sample_ticker = df["ticker"].value_counts().index[0]
d = df[df["ticker"] == sample_ticker].sort_values("date").copy()

plt.figure(figsize=(14, 5))
plt.plot(d["date"], d["close"], label="Close")
for ma in ["ma_20", "ma_50", "ma_200"]:
    if ma in d.columns:
        plt.plot(d["date"], d[ma], label=ma)
plt.title(f"Kurs und Moving Averages – {sample_ticker}")
plt.legend()
plt.show()

In [ ]:
if "drawdown" in d.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(d["date"], d["drawdown"] * 100)
    plt.title(f"Drawdown – {sample_ticker}")
    plt.ylabel("Drawdown in %")
    plt.show()

In [ ]:
if "volatility_20d" in d.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(d["date"], d["volatility_20d"] * 100)
    plt.title(f"20T Volatilität – {sample_ticker}")
    plt.ylabel("Volatilität in %")
    plt.show()

## Ergebnis

Die Features sind fachlich gruppierbar und können in der App verständlich erklärt werden.